# Making hazard curves

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *

## Specify the set of events and the annual probability of each

Here we use the Buried events with weights from that part of the logic tree (which should sum to 0.75).
We then compute these to annual probabilities based on a 1/526 probability of a "CSZ event" happening.

Note that the list of events and weights can be obtained easily using the `CHTtools` module, see  
- https://depts.washington.edu/ptha/CHTuser/dtopo/chttoolsdemo/

In [ ]:
event_weights = \
{'BL10D': 0.0375,
 'BL10M': 0.0625,
 'BL10S': 0.025,
 'BL13D': 0.0375,
 'BL13M': 0.0625,
 'BL13S': 0.025,
 'BL16D': 0.0375,
 'BL16M': 0.0625,
 'BL16S': 0.025,
 'BR10D': 0.0375,
 'BR10M': 0.0625,
 'BR10S': 0.025,
 'BR13D': 0.0375,
 'BR13M': 0.0625,
 'BR13S': 0.025,
 'BR16D': 0.0375,
 'BR16M': 0.0625,
 'BR16S': 0.025}

# create numpy array of event names for sorting below:
events = array(list(event_weights.keys()), dtype=str)

weights = array([event_weights[e] for e in events])
print(f'The weights sum to {weights.sum()}, (omitting FrontalThrust events)')

# annual probability of each event occuring:
#event_probs = weights / 526         # WRONG in quadratic terms
event_probs = 1 - exp(-weights/526)  # more correct for Poisson process

print('\nAnnual probabilities:')
for k in range(len(events)):
    print(f'{events[k]}: {event_probs[k]:.10f}  compare to wrong way: {weights[k]/526:.10f}')

## Sample data (hmax at one gauge location for all events)

This data came from one particular tide gauge at a site in Grays Harbor (Gauge 431 near the tip of Ocean Shores).

In [ ]:
hmax = array([6.1590548e+00, 7.5819397e-01, 2.9979393e+00, 1.3236976e+00,
       1.2181695e+00, 1.3012362e-01, 5.4883194e-01, 3.1585693e-03,
       7.0573425e-01, 2.5214369e+00, 1.4575415e+00, 6.2790704e-01,
       2.5484571e+00, 7.0056701e-01, 3.7961149e-01, 8.3777404e-01,
       4.7398233e-01, 4.1361332e-01])

## Plot these values for each event

In [ ]:
ievents = range(len(events))
plot(hmax, ievents)
yticks(ievents, events);
grid(True)
xlabel('meters')
title(f'Maximum depth h');

## Sort from smallest to largest hmax

This gives a nicer plot and is needed for creating hazard curves.

In [ ]:
ind = argsort(hmax)
hmax_sort = hmax[ind]
events_sort = events[ind]
probs_sort = event_probs[ind]

In [ ]:
ievents = range(len(events)-1,-1,-1)
plot(hmax_sort, ievents)
yticks(ievents, events_sort);
grid(True)
xlabel('meters')
title(f'Maximum depth h (after sorting events by h)');

## Plot as a bar chart within indication of relative probabilities

In [ ]:
heights_sort = 7000 * probs_sort  # scale to get reasonable height plot
fig,ax = subplots()
ax.barh(events_sort,hmax_sort,height=heights_sort)
ax.invert_yaxis()
ax.set_title('hmax for each event, bar width proportional to probability');

## Make hazard curve

Compute the cumulative weights `pcum[k]` is the probability of at least one event happening that meets or exceeds the depth `hmax_sorted[k]`.

In [ ]:
pcum = zeros(len(hmax_sort)+1)  # with one extra 0 to start sum at end, and for plotting

for k in range(len(events_sort)-1,-1,-1):
    pcum[k] = pcum[k+1] + probs_sort[k] - pcum[k+1] * probs_sort[k]

# add one more data point with h > hmax.max() and probability 0, for tail of step function:
hmax_step = hstack((hmax_sort, hmax_sort.max()+1))

## Plot hazard curve as step function

In [ ]:
step(hmax_step, pcum, 'b', where='pre')
xlim(0, hmax_step.max())
grid(True)
title(f'Hazard curve for {len(events)} events');

## Make nicer side by side plots

In [ ]:
fig,axs = subplots(1,2,figsize=(12,5))
ax = axs[0]
ax.barh(events_sort,hmax_sort,height=heights_sort)
ax.invert_yaxis()
ax.set_xlim(0, hmax_step.max())
ax.set_xlabel('Max water depth of each event at specified location')
ax.set_title('hmax for each event, bar width proportional to probability');

ax = axs[1]
ax.step(hmax_step, pcum, 'b', where='pre')
ax.set_xlim(-0.1, hmax_step.max())
ax.set_xlabel('Exceedence values for max water depth')
ax.set_ylabel('Annual probability of at least one event exceeding')
ax.grid(True)
ax.set_title(f'Hazard curve for {len(events)} events');
ax.plot(hmax_step, (1/2475.)*ones(hmax_step.shape), 'r--',label='1/2475')
ax.plot(hmax_step, (1/975.)*ones(hmax_step.shape), 'c--',label='1/975')
ax.legend(loc='upper right', framealpha=1);

fname = 'SampleHazardCurve.png'
savefig(fname)

## Check that hazard curve approaches expected probability 1/526 at y-axis

In [ ]:
print(f'total probability = pcum[0]            = {pcum[0]:.9f}')
print(f'  should agree with 1 - exp(-0.75/526) = {1 - exp(-0.75/526):.9f}')

In [ ]:
#print(f'hmax_step = \n{hmax_step}\npcum = \n{pcum}')

## Print table to compare against results from Loyce's code

In [ ]:
for k in range(len(hmax_step)):
    print(f'{hmax_step[k]:.2f}  {pcum[k]:.9f}')